In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

In [3]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

In [7]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [8]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [9]:
df['industry'].mode()

0    technology
Name: industry, dtype: object

In [10]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [11]:
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42
)

In [12]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [13]:
y_full_train = df_full_train.converted.values

In [14]:
del df_full_train['converted']

In [15]:
df_train.shape, df_val.shape, df_test.shape

((3000, 8), (1000, 8), (1000, 8))

In [16]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, y_train)

In [17]:
mi = df_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [18]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, y_train)

In [19]:
y_train

array([1, 0, 1, ..., 1, 0, 0])

In [20]:
df_train[categorical].apply(mutual_info_churn_score)

lead_source          0.034346
industry             0.001995
employment_status    0.018966
location             0.001043
dtype: float64

In [21]:
#one hot encoding

dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [22]:
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [23]:
y_pred = model.predict(X_val)

In [24]:
accuracy = accuracy_score(y_val, y_pred)
round(accuracy, 2)

0.65

In [25]:
features_to_test = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

In [26]:
original_accuracy = accuracy

for feature in features_to_test:
    df_train_small = df_train.drop(columns=[feature])
    df_val_small = df_val.drop(columns=[feature])

    dv_small = DictVectorizer(sparse=False)

    train_dicts = df_train_small.to_dict(orient='records')
    X_train_small = dv_small.fit_transform(train_dicts)

    val_dicts = df_val_small.to_dict(orient='records')
    X_val_small = dv_small.transform(val_dicts)

    model_small = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_small.fit(X_train_small, y_train)

    y_pred_small = model_small.predict(X_val_small)

    accuracy_small = accuracy_score(y_val, y_pred_small)

    difference = original_accuracy - accuracy_small

    print(
        feature,
        "accuracy:", accuracy_small,
        "difference:", difference
    )

lead_source accuracy: 0.642 difference: 0.0030000000000000027
number_of_courses_viewed accuracy: 0.643 difference: 0.0020000000000000018
interaction_count accuracy: 0.601 difference: 0.04400000000000004


In [27]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    score = accuracy_score(y_val, y_pred)

    print(C, round(score, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
